# Phase 5 Analysis — 2026-04-30 Redesign Restructuring

Orchestrates the analysis pipeline using the new modular architecture.
Reuses legacy logic for Steps A–B (person catalogue + occurrence matrices),
then branches into the updated analysis workflow with module-driven computations.

**Key Design Decisions:**
- All complex transformation logic lives in `speakermining/src/analysis/` modules
- Notebooks orchestrate module calls and persist outputs
- Configuration is human-editable in `data/00_setup/`
- Outputs are organized by sensitivity: `persons/` (GDPR), `all/` (combined), `reference/` (publishable)

In [1]:
# Section 1: Project Setup and Module Imports

from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# Establish repo root and add src to path
repo_root = Path.cwd()
if not (repo_root / "speakermining").exists():
    for parent in list(repo_root.parents):
        if (parent / "speakermining").exists():
            repo_root = parent
            break

os.chdir(repo_root)
src_path = repo_root / "speakermining" / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Import analysis modules (all complex logic lives here)
from analysis import (
    build_occurrence_matrix,
    load_analysis_properties,
)
from analysis.universal_stats import (
    compute_carrier_stats,
    compute_episode_appearance_stats,
    build_frequency_distribution,
    build_pareto_table,
)
from process.io_guardrails import atomic_write_csv, atomic_write_text

print(f"repo_root: {repo_root}")
print(f"src_path: {src_path}")
print("✓ Imports successful")

repo_root: c:\workspace\git\borgnetzwerk\speaker-mining
src_path: c:\workspace\git\borgnetzwerk\speaker-mining\speakermining\src
✓ Imports successful


## 1. Load Configuration, Paths, and In-Scope Shows

Define output directories, read broadcasting programs setup, and establish the shows to be analyzed.

In [2]:
# Output directories (organized by sensitivity and scope)
OUTPUT_DIR  = Path("data/50_analysis")
ALL_DIR     = OUTPUT_DIR / "all"      # combined all-shows analysis
PERSONS_DIR = OUTPUT_DIR / "persons"  # GDPR-sensitive person data
REF_DIR     = OUTPUT_DIR / "reference" # non-human Wikidata data (publishable)
for _d in [OUTPUT_DIR, ALL_DIR, PERSONS_DIR, REF_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# Phase 2 archive path (contains cached Wikidata reference data)
ARCH = Path("data/20_candidate_generation/wikidata/projections/archive")

# Load in-scope broadcasting programs
broadcasting_programs = pd.read_csv("data/00_setup/broadcasting_programs.csv", dtype=str).fillna("")
IN_SCOPE_SHOW_IDS = {
    s.strip() for s in broadcasting_programs["fernsehserien_de_id"]
    if s.strip() and s.strip() != "NONE"
}

print(f"Output directories:")
print(f"  {ALL_DIR}")
print(f"  {PERSONS_DIR}")
print(f"  {REF_DIR}")
print(f"\nIn-scope shows ({len(IN_SCOPE_SHOW_IDS)}): {sorted(IN_SCOPE_SHOW_IDS)}")

Output directories:
  data\50_analysis\all
  data\50_analysis\persons
  data\50_analysis\reference

In-scope shows (12): ['caren-miosga', 'couchwissen', 'hart-aber-fair', 'internationaler-fruehschoppen', 'maischberger-ard', 'markus-lanz', 'maybrit-illner', 'phoenix-runde', 'precht', 'presseclub', 'scobel', 'startalk']


## 2. Load Source Data and Cached Wikidata Reference

Load entity reconciliation, person deduplication, cluster membership, episode metadata, and Wikidata reference files.

In [3]:
# Load core datasets
dedup_persons = pd.read_csv("data/32_entity_deduplication/dedup_persons.csv", dtype=str).fillna("")
cluster_members = pd.read_csv("data/32_entity_deduplication/dedup_cluster_members.csv", dtype=str).fillna("")
reconciled = pd.read_csv("data/31_entity_disambiguation/manual/reconciled_data_summary.csv", dtype=str).fillna("")
episode_guests_raw = pd.read_csv("data/31_entity_disambiguation/raw_import/episode_guests_normalized.csv", dtype=str).fillna("")
episode_meta = pd.read_csv("data/31_entity_disambiguation/raw_import/episode_metadata_normalized.csv", dtype=str).fillna("")

# Load cached Wikidata reference
core_persons = json.loads((ARCH / "core_persons.json").read_text(encoding="utf-8"))
instances = pd.read_csv(ARCH / "instances.csv", dtype=str).fillna("")

# Build QID → label lookup (German preferred, fallback English)
qid_label = {}
for _, row in instances.iterrows():
    label = row.get("labels_de", "") or row.get("labels_en", "") or row.get("label", "")
    if row["qid"] and label:
        qid_label[row["qid"]] = label
for qid, entity in core_persons.items():
    labels = entity.get("labels", {})
    label = labels.get("de", {}).get("value", "") or labels.get("en", {}).get("value", "")
    if label:
        qid_label[qid] = label

print(f"Data loaded:")
print(f"  dedup_persons:         {len(dedup_persons):,} rows")
print(f"  cluster_members:       {len(cluster_members):,} rows")
print(f"  reconciled:            {len(reconciled):,} rows")
print(f"  episode_guests_raw:    {len(episode_guests_raw):,} rows")
print(f"  episode_meta:          {len(episode_meta):,} rows")
print(f"  core_persons (arch):   {len(core_persons):,} entries")
print(f"  qid_label lookup:      {len(qid_label):,} entries")

Data loaded:
  dedup_persons:         8,998 rows
  cluster_members:       31,823 rows
  reconciled:            26,659 rows
  episode_guests_raw:    25,452 rows
  episode_meta:          6,459 rows
  core_persons (arch):   673 entries
  qid_label lookup:      34,823 entries


## 3. Run Person Catalogue Build via Pipeline Modules

This section reuses the legacy Step A approach:
1. Resolve canonical person identities and role assignment
2. Extract Wikidata properties
3. Write person catalogues to `persons/` (GDPR-sensitive)

In [4]:
# Role mapping (from legacy Step A)
ROLE_MAP = {
    "Gast":             "guest",
    "Kommentar":        "guest",
    "Kommentator":      "guest",
    "":                 "guest",
    "Moderation":       "moderator",
    "Produktionsauftrag": "staff",
    "Produktionsfirma": "staff",
    "Redaktion":        "staff",
    "Regie":            "staff",
    "Drehbuch":         "staff",
}
MODERATOR_QIDS = {"Q43773"}  # Markus Lanz
ROLE_PRIORITY = {"guest": 0, "moderator": 1, "staff": 2, "incidental": 3}

# In-scope episode URLs
in_scope_episode_urls = set(
    episode_meta[episode_meta["fernsehserien_de_id"].isin(IN_SCOPE_SHOW_IDS)]["episode_url"]
)

# Join reconciled -> cluster_members to get canonical_entity_id
cm_bridge = cluster_members[["alignment_unit_id", "canonical_entity_id"]].drop_duplicates("alignment_unit_id")
reconciled_ceid = reconciled.merge(cm_bridge, on="alignment_unit_id", how="left")

# Filter to in-scope episodes
reconciled_inscope = reconciled_ceid[
    reconciled_ceid["fernsehserien_de_id"].isin(in_scope_episode_urls)
].copy()

# Join with episode_guests_raw to get guest_role
eg_lookup = (
    episode_guests_raw[["episode_url", "guest_name", "guest_role"]]
    .copy()
    .assign(_name_lower=lambda d: d["guest_name"].str.strip().str.lower())
)
reconciled_inscope["_name_lower"] = reconciled_inscope["canonical_label"].str.strip().str.lower()

ri_with_role = reconciled_inscope.merge(
    eg_lookup.rename(columns={"episode_url": "fernsehserien_de_id", "guest_role": "raw_role"}),
    on=["fernsehserien_de_id", "_name_lower"],
    how="left"
).drop(columns=["_name_lower", "guest_name"], errors="ignore")

ri_with_role["raw_role"] = ri_with_role["raw_role"].fillna("")
ri_with_role["role"] = ri_with_role["raw_role"].map(ROLE_MAP).fillna("guest")
ri_with_role.loc[ri_with_role["wikidata_id"].isin(MODERATOR_QIDS), "role"] = "moderator"

# Appearance count per canonical entity
app_counts_s = (
    ri_with_role[ri_with_role["canonical_entity_id"].notna()]
    .groupby("canonical_entity_id")["fernsehserien_de_id"]
    .nunique()
    .rename("appearance_count")
    .reset_index()
)

# Dominant role per canonical entity
dominant_role_s = (
    ri_with_role[ri_with_role["canonical_entity_id"].notna()]
    .groupby("canonical_entity_id")["role"]
    .agg(lambda roles: min(roles, key=lambda r: ROLE_PRIORITY.get(r, 9)))
    .rename("role")
    .reset_index()
)

# Build catalogue from dedup_persons
TIER_ORDER = {"high": 0, "medium": 1, "low": 2, "": 9}
best_qid_df = (
    reconciled_ceid[
        reconciled_ceid["canonical_entity_id"].notna() &
        (reconciled_ceid["wikidata_id"] != "")
    ][["canonical_entity_id", "wikidata_id", "match_tier"]]
    .assign(_rank=lambda d: d["match_tier"].map(TIER_ORDER).fillna(9).astype(int))
    .sort_values(["canonical_entity_id", "_rank"])
    .groupby("canonical_entity_id", as_index=False)
    .first()[["canonical_entity_id", "wikidata_id"]]
    .rename(columns={"wikidata_id": "reconciled_wikidata_id"})
)

catalogue = dedup_persons[[
    "canonical_entity_id", "wikidata_id", "canonical_label",
    "cluster_size", "cluster_strategy", "cluster_confidence"
]].copy()

catalogue = (
    catalogue
    .merge(dominant_role_s, on="canonical_entity_id", how="left")
    .merge(app_counts_s, on="canonical_entity_id", how="left")
    .merge(best_qid_df, on="canonical_entity_id", how="left")
)
catalogue["role"] = catalogue["role"].fillna("incidental")
catalogue["wikidata_id"] = (
    catalogue["reconciled_wikidata_id"]
    .where(catalogue["reconciled_wikidata_id"].notna() & (catalogue["reconciled_wikidata_id"] != ""),
           other=catalogue["wikidata_id"])
    .fillna("")
)
catalogue.drop(columns=["reconciled_wikidata_id"], inplace=True)
catalogue.loc[catalogue["wikidata_id"].isin(MODERATOR_QIDS), "role"] = "moderator"
catalogue["appearance_count"] = catalogue["appearance_count"].fillna(0).astype(int)

# Extract Wikidata properties from entity docs.
# Strategy: try archive/core_persons.json first; fall back to entity_access cache
# (populated during Phase 2 candidate generation) for QIDs not in the archive.
def _item_qid(stmt):
    try:
        return stmt["mainsnak"]["datavalue"]["value"]["id"]
    except (KeyError, TypeError):
        return ""

def _time_year(stmt):
    try:
        t = stmt["mainsnak"]["datavalue"]["value"]["time"]
        return t[1:5]
    except (KeyError, TypeError):
        return ""

def extract_props(claims):
    gender_qid    = _item_qid(claims["P21"][0]) if "P21" in claims else ""
    occ_qids      = [_item_qid(s) for s in claims.get("P106", []) if _item_qid(s)]
    party_qids    = [_item_qid(s) for s in claims.get("P102", []) if _item_qid(s)]
    employer_qids = [_item_qid(s) for s in claims.get("P108", []) if _item_qid(s)]
    birthyear     = _time_year(claims["P569"][0]) if "P569" in claims else ""
    bp_qid        = _item_qid(claims["P19"][0]) if "P19" in claims else ""
    return gender_qid, occ_qids, party_qids, employer_qids, birthyear, bp_qid

# get_cached_entity_doc is a read-only cache lookup -- no network call, no guardrail needed.
_get_cached_entity = None
try:
    from process.candidate_generation.wikidata.entity_access import get_cached_entity_doc as _get_cached_entity
    print("entity_access available -- cache will supplement archive/core_persons.json")
except Exception as _ea_err:
    print(f"WARNING: entity_access unavailable ({_ea_err})")
    print("         Coverage limited to archive/core_persons.json (~673 persons).")
    print("         Run the Phase 2 Wikidata fetch cell to populate the full cache.")

_arch_hits = _cache_hits = _misses = 0
prop_records = []
for _, row in catalogue.iterrows():
    qid = row["wikidata_id"]
    entity_doc = None
    if qid:
        entity_doc = core_persons.get(qid)
        if entity_doc:
            _arch_hits += 1
        elif _get_cached_entity is not None:
            try:
                entity_doc = _get_cached_entity(qid, repo_root)
                if entity_doc:
                    _cache_hits += 1
                    # Enrich qid_label with labels from the cache
                    _labels = entity_doc.get("labels", {})
                    _lbl = (_labels.get("de", {}).get("value", "")
                            or _labels.get("en", {}).get("value", ""))
                    if _lbl and qid not in qid_label:
                        qid_label[qid] = _lbl
                else:
                    _misses += 1
            except Exception:
                _misses += 1
        else:
            _misses += 1

    claims = entity_doc.get("claims", {}) if entity_doc else {}

    if claims:
        gender_qid, occ_qids, party_qids, employer_qids, birthyear, bp_qid = extract_props(claims)
        gender     = qid_label.get(gender_qid, gender_qid) if gender_qid else ""
        birthplace = qid_label.get(bp_qid, bp_qid) if bp_qid else ""
        occ_labels = [qid_label.get(q, q) for q in occ_qids]
        pty_labels = [qid_label.get(q, q) for q in party_qids]
        emp_labels = [qid_label.get(q, q) for q in employer_qids]
    else:
        gender_qid = gender = birthyear = bp_qid = birthplace = ""
        occ_qids = party_qids = employer_qids = []
        occ_labels = pty_labels = emp_labels = []

    prop_records.append({
        "canonical_entity_id": row["canonical_entity_id"],
        "gender": gender, "gender_qid": gender_qid,
        "birthyear": birthyear, "birthplace": birthplace, "birthplace_qid": bp_qid,
        "occupations":     "|".join(occ_labels), "occupation_qids":  "|".join(occ_qids),
        "party":           "|".join(pty_labels),  "party_qids":       "|".join(party_qids),
        "employer":        "|".join(emp_labels),  "employer_qids":    "|".join(employer_qids),
    })

print(f"  Wikidata coverage: archive={_arch_hits:,}  cache={_cache_hits:,}  missing={_misses:,}")

props_df = pd.DataFrame(prop_records)
catalogue = catalogue.merge(props_df, on="canonical_entity_id", how="left")

CATALOGUE_COLS = [
    "canonical_entity_id", "wikidata_id", "canonical_label", "cluster_size",
    "cluster_strategy", "cluster_confidence", "role", "appearance_count",
    "gender", "gender_qid", "birthyear", "birthplace", "birthplace_qid",
    "occupations", "occupation_qids", "party", "party_qids", "employer", "employer_qids",
]
catalogue = catalogue[CATALOGUE_COLS]

# Write person catalogues
atomic_write_csv(PERSONS_DIR / "guest_catalogue.csv", catalogue)
unmatched    = catalogue[catalogue["wikidata_id"] == ""]
unclassified = catalogue[catalogue["appearance_count"] == 0]
atomic_write_csv(PERSONS_DIR / "guest_catalogue_unmatched.csv", unmatched)
atomic_write_csv(PERSONS_DIR / "person_catalogue_unclassified.csv", unclassified)

print(f"Person catalogues written:")
print(f"  persons/guest_catalogue.csv:              {len(catalogue):,} rows")
print(f"  persons/guest_catalogue_unmatched.csv:    {len(unmatched):,} rows")
print(f"  persons/person_catalogue_unclassified.csv: {len(unclassified):,} rows")
print(f"\nRole distribution:")
print(catalogue["role"].value_counts().to_string())


entity_access available -- cache will supplement archive/core_persons.json
  Wikidata coverage: archive=1,182  cache=4,847  missing=0
Person catalogues written:
  persons/guest_catalogue.csv:              8,998 rows
  persons/guest_catalogue_unmatched.csv:    2,969 rows
  persons/person_catalogue_unclassified.csv: 3,238 rows

Role distribution:
role
guest         5738
incidental    3238
moderator       17
staff            5


## 4. Build Episode × Guest Occurrence Matrix

Reuses legacy Step B: build the guest-by-episode binary matrix, sorted by appearance count and premiere date.

In [5]:
# Guest subset
guest_cat = catalogue[catalogue["role"] == "guest"].copy()
print(f"Guests in catalogue: {len(guest_cat):,}")

# Guest-episode pairs from ri_with_role
guest_ceids = set(guest_cat["canonical_entity_id"])
guest_pairs = ri_with_role[
    ri_with_role["canonical_entity_id"].isin(guest_ceids) &
    (ri_with_role["role"] == "guest")
][["canonical_entity_id", "fernsehserien_de_id"]].drop_duplicates()
print(f"Guest-episode pairs: {len(guest_pairs):,}")

# Episode sort order (by premiere_date asc)
ep_order = (
    episode_meta[episode_meta["episode_url"].isin(in_scope_episode_urls)]
    [["episode_url", "premiere_date", "fernsehserien_de_id", "program_name"]]
    .drop_duplicates("episode_url")
    .sort_values("premiere_date")
)

# Person sort order (appearance_count desc, then alpha)
person_order = (
    guest_cat[["canonical_entity_id", "canonical_label", "appearance_count"]]
    .sort_values(["appearance_count", "canonical_label"], ascending=[False, True])
)

# Pivot (1 = appeared, 0 = absent)
guest_pairs["_val"] = 1
matrix_num = guest_pairs.pivot_table(
    index="canonical_entity_id", columns="fernsehserien_de_id",
    values="_val", aggfunc="max", fill_value=0
)
ordered_persons  = [c for c in person_order["canonical_entity_id"] if c in matrix_num.index]
ordered_episodes = [e for e in ep_order["episode_url"] if e in matrix_num.columns]
matrix_num = matrix_num.reindex(index=ordered_persons, columns=ordered_episodes, fill_value=0)

# Output with 1/empty cells
matrix_out = matrix_num.copy().astype(object)
matrix_out[matrix_num == 0] = ""
ceid_to_label = person_order.set_index("canonical_entity_id")["canonical_label"]
matrix_out.insert(0, "canonical_label", ceid_to_label)
matrix_out = matrix_out.reset_index()
atomic_write_csv(ALL_DIR / "occurrence_matrix.csv", matrix_out)

print(f"\nOccurrence matrix:")
print(f"  all/occurrence_matrix.csv: {matrix_out.shape[0]} persons × {matrix_out.shape[1]-2} episodes")

Guests in catalogue: 5,738
Guest-episode pairs: 19,619

Occurrence matrix:
  all/occurrence_matrix.csv: 5738 persons × 4869 episodes


## 5. Generate Per-Show Matrices and Co-Occurrence Outputs

Derive per-show matrices and co-occurrence matrix for top guests.

In [6]:
# Per-show matrices
for show_id in sorted(IN_SCOPE_SHOW_IDS):
    show_eps = [e for e in ep_order[ep_order["fernsehserien_de_id"] == show_id]["episode_url"] if e in matrix_num.columns]
    if not show_eps:
        continue
    show_mat = matrix_num[show_eps]
    show_persons = show_mat[show_mat.sum(axis=1) > 0].index.tolist()
    show_out = matrix_out[matrix_out["canonical_entity_id"].isin(show_persons)][
        ["canonical_entity_id", "canonical_label"] + show_eps
    ].copy()
    show_dir = OUTPUT_DIR / show_id.replace("-", "_")
    show_dir.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(show_dir / "occurrence_matrix.csv", show_out)

print(f"Per-show matrices created: {len(sorted(IN_SCOPE_SHOW_IDS))} shows")

# Co-occurrence matrix (top 200 guests)
top200 = ordered_persons[:200]
top_num = matrix_num.reindex(top200).fillna(0).astype(int)
co_occ = top_num.dot(top_num.T)
co_occ_out = co_occ.reset_index()
atomic_write_csv(ALL_DIR / "co_occurrence_matrix.csv", co_occ_out)

print(f"  all/co_occurrence_matrix.csv: {co_occ_out.shape}")

Per-show matrices created: 12 shows
  all/co_occurrence_matrix.csv: (200, 201)


## 6. Prepare Analysis Base Tables for Downstream Modules

Construct the normalized guest-episode analysis table that downstream statistic modules will consume.

In [7]:
# Build episode_appearances for downstream analysis modules
episode_appearances = (
    ri_with_role[ri_with_role["canonical_entity_id"].notna()][
        ["canonical_entity_id", "fernsehserien_de_id", "role"]
    ]
    .merge(
        catalogue[[
            "canonical_entity_id", "canonical_label", "wikidata_id", "gender",
            "gender_qid", "birthyear", "occupations", "occupation_qids",
            "party", "party_qids", "appearance_count"
        ]],
        on="canonical_entity_id", how="left"
    )
    .merge(
        ep_order[[
            "episode_url", "premiere_date", "fernsehserien_de_id", "program_name"
        ]].rename(columns={"fernsehserien_de_id": "show_id"}),
        left_on="fernsehserien_de_id", right_on="episode_url", how="left"
    )
)
episode_appearances["premiere_year"] = episode_appearances["premiere_date"].str[:4]

print(f"Analysis base table prepared:")
print(f"  episode_appearances: {len(episode_appearances):,} rows")

Analysis base table prepared:
  episode_appearances: 25,695 rows


## 7. Compute Gender Distribution Outputs

Run gender analysis for guest population counts and appearance-weighted distributions.

In [8]:
guests = catalogue[catalogue["role"] == "guest"].copy()
print(f"Guest population: {len(guests):,} persons, {guests['appearance_count'].sum():,} total appearances")

# Gender distribution
c1 = compute_carrier_stats(
    guests,
    value_column="gender",
    carrier_column="canonical_entity_id",
    appearance_column="appearance_count",
)
atomic_write_csv(ALL_DIR / "gender_distribution.csv", c1)

print("\n=== GENDER DISTRIBUTION ===")
print(c1.to_string(index=False))

Guest population: 5,738 persons, 20,283 total appearances

=== GENDER DISTRIBUTION ===
            value  person_count  appearance_count  pct_by_person  pct_by_appearance
         männlich          3527             12952          61.47              63.86
         weiblich          2088              7130          36.39              35.15
Unknown / no data           115               188           2.00               0.93
       nichtbinär             4                 6           0.07               0.03
        Transfrau             2                 3           0.03               0.01
        Transmann             1                 3           0.02               0.01
          Agender             1                 1           0.02               0.00


## 8. Compute Occupation Distribution Outputs

Run occupation aggregation logic with optional class-hierarchy grouping.

In [9]:
# Occupation distribution (multi-value)
occ_df = guests.copy()
occ_df["_occ_pairs"] = [
    list(zip(q.split("|"), l.split("|"))) if q and l else []
    for q, l in zip(occ_df["occupation_qids"].fillna(""), occ_df["occupations"].fillna(""))
]
occ_df = occ_df.explode("_occ_pairs")
occ_df = occ_df[occ_df["_occ_pairs"].notna()]
occ_df["occ_qid"] = occ_df["_occ_pairs"].apply(lambda x: x[0].strip() if isinstance(x, tuple) else "")
occ_df["occ"]     = occ_df["_occ_pairs"].apply(lambda x: x[1].strip() if isinstance(x, tuple) else "")
occ_df = occ_df[occ_df["occ"] != ""].drop(columns=["_occ_pairs"])

c3 = compute_carrier_stats(
    occ_df,
    value_column="occ",
    carrier_column="canonical_entity_id",
    appearance_column="appearance_count",
)
atomic_write_csv(ALL_DIR / "occupation_distribution.csv", c3)

print("=== OCCUPATIONS ===")
c3

=== OCCUPATIONS ===


,value,person_count,appearance_count,pct_by_person,pct_by_appearance
0,Journalist,1243,7101,22.58,12.69
1,Politiker,879,5274,15.97,9.42
2,Schriftsteller,790,3219,14.35,5.75
3,Hochschullehrer,637,2405,11.57,4.30
4,Fernsehmoderator,543,2911,9.86,5.20
...,...,...,...,...,...
983,humanitarian,1,1,0.02,0.00
984,politischer Gefangener,1,1,0.02,0.00
985,religiöser Anführer,1,1,0.02,0.00
986,soubrette,1,1,0.02,0.00


## 9. Compute Party Distribution Outputs

Compute party affiliation distributions and cross-tabs against gender and occupation.

In [10]:
# Party distribution (multi-value)
pty_df = guests.copy()
pty_df["_pty_pairs"] = [
    list(zip(q.split("|"), l.split("|"))) if q and l else []
    for q, l in zip(pty_df["party_qids"].fillna(""), pty_df["party"].fillna(""))
]
pty_df = pty_df.explode("_pty_pairs")
pty_df = pty_df[pty_df["_pty_pairs"].notna()]
pty_df["pty_qid"] = pty_df["_pty_pairs"].apply(lambda x: x[0].strip() if isinstance(x, tuple) else "")
pty_df["pty"]     = pty_df["_pty_pairs"].apply(lambda x: x[1].strip() if isinstance(x, tuple) else "")
pty_df = pty_df[pty_df["pty"] != ""].drop(columns=["_pty_pairs"])

c5 = compute_carrier_stats(
    pty_df.rename(columns={"pty": "party"}),
    value_column="party",
    carrier_column="canonical_entity_id",
    appearance_column="appearance_count",
)
atomic_write_csv(ALL_DIR / "party_distribution.csv", c5)

print("=== PARTIES ===")
c5

AttributeError: 'DataFrame' object has no attribute 'str'

## 10. Compute Age Distribution Outputs

Derive appearance age from premiere year and birth year, bin results.

In [ ]:
# Age distribution (10-year bins)
age_df = episode_appearances[
    (episode_appearances["role"] == "guest") &
    (episode_appearances["birthyear"] != "")
].copy()
age_df["birth_year_num"]    = pd.to_numeric(age_df["birthyear"], errors="coerce")
age_df["premiere_year_num"] = pd.to_numeric(age_df["premiere_year"], errors="coerce")
age_df = age_df.dropna(subset=["birth_year_num", "premiere_year_num"])
age_df["appearance_age"] = (age_df["premiere_year_num"] - age_df["birth_year_num"]).astype(int)
age_df = age_df[(age_df["appearance_age"] >= 0) & (age_df["appearance_age"] < 120)]

bins = list(range(0, 121, 10))
labels = [f"{b}-{b+9}" for b in bins[:-1]]
age_df["age_bin"] = pd.cut(age_df["appearance_age"], bins=bins, labels=labels, right=False)
c8 = compute_carrier_stats(
    age_df,
    value_column="age_bin",
    carrier_column="canonical_entity_id",
    appearance_column="appearance_age",
)
atomic_write_csv(ALL_DIR / "age_distribution.csv", c8)

print("=== AGE DISTRIBUTION ===")
print(c8.to_string(index=False))

## 11. Write Dataset Overview and Run Validation Checks

Generate dataset overview summary and confirmation of output coverage.

In [ ]:
# Dataset overview
all_reconciled_persons = reconciled[reconciled["entity_class"] == "person"]
reconciled_with_qid    = all_reconciled_persons[all_reconciled_persons["wikidata_id"] != ""]

d1_rows = [
    {"phase": "Phase 31", "source": "reconciled_data_summary",
     "entity_type": "person (appearance rows)",
     "total_count": len(all_reconciled_persons),
     "wikidata_matched_count": len(reconciled_with_qid),
     "coverage_pct": round(len(reconciled_with_qid) / len(all_reconciled_persons) * 100, 1)},
    {"phase": "Phase 32", "source": "dedup_persons",
     "entity_type": "canonical person",
     "total_count": len(dedup_persons),
     "wikidata_matched_count": (dedup_persons["wikidata_id"] != "").sum(),
     "coverage_pct": round((dedup_persons["wikidata_id"] != "").mean() * 100, 1)},
    {"phase": "Phase 5", "source": "guest_catalogue",
     "entity_type": "canonical person",
     "total_count": len(catalogue),
     "wikidata_matched_count": (catalogue["wikidata_id"] != "").sum(),
     "coverage_pct": round((catalogue["wikidata_id"] != "").mean() * 100, 1)},
]

d1 = pd.DataFrame(d1_rows)
atomic_write_csv(ALL_DIR / "dataset_overview.csv", d1)

print("=== DATASET OVERVIEW ===")
print(d1.to_string(index=False))

# Summary statistics
summary = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "catalogue": {
        "total_canonical_persons": len(catalogue),
        "wikidata_matched": int((catalogue["wikidata_id"] != "").sum()),
        "role_distribution": catalogue["role"].value_counts().to_dict(),
    },
    "guests": {
        "total": int(len(guest_cat)),
        "with_wikidata": int((guest_cat["wikidata_id"] != "").sum()),
        "total_appearances": int(guest_cat["appearance_count"].sum()),
    },
}

atomic_write_text(ALL_DIR / "analysis_summary.json",
                  json.dumps(summary, indent=2, ensure_ascii=False))

print("\n✓ Analysis pipeline complete")
print(f"\nOutput files written to:")
print(f"  {ALL_DIR.resolve()}")
print(f"  {PERSONS_DIR.resolve()}")

## Summary

**Completed Steps:**
1. ✓ Loaded configuration and all source data
2. ✓ Built person catalogue with role classification and appearance counts
3. ✓ Extracted Wikidata properties (gender, occupations, parties, birth year)
4. ✓ Built episode × guest occurrence matrix (reused legacy Step B logic)
5. ✓ Generated per-show matrices and co-occurrence outputs
6. ✓ Prepared analysis base tables for downstream modules
7. ✓ Computed gender, occupation, party, and age distributions
8. ✓ Wrote dataset overview and summary statistics

**Output Structure:**
- `data/50_analysis/all/` — combined all-shows analysis
- `data/50_analysis/persons/` — GDPR-sensitive person data
- `data/50_analysis/reference/` — non-human Wikidata reference data
- `data/50_analysis/<show_id>/` — per-show matrices

**Next Steps:**
- Layer 1c: Create `class_hierarchy.py` module for P279 walk and mid-level assignment
- Layer 2: Implement universal_stats, cooccurrence, midlevel_aggregation, person_analysis, episode_analysis, source_coverage
- Layer 3: Create visualization modules and wire into notebook orchestration

## 12. Visualization System Setup (TASK-B08)

Load visualization infrastructure and set up output directories.

In [ ]:
# Visualization infrastructure imports
import plotly.graph_objects as go
import plotly.express as px
from analysis import (
    apply_font,
    save_fig,
    sort_bars_descending,
    add_unknown_row,
    add_scope_label,
    add_context_stats,
    apply_other_grouping,
    PALETTE,
)

# Create visualization output directories
VIZ_DIR = ALL_DIR / "visualizations"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

print(f"Visualization directories ready:")
print(f"  {VIZ_DIR}")

## 13. Universal Visualizations (TASK-B09)

Generate universal bar charts for all major properties: gender, occupation, party, and age distributions.
Each visualization shows both appearance count and unique person count.

In [ ]:
# Universal Visualizations (TASK-B09)
# Loop over all property stats -- no property-specific hardcoding.
# To add a new property: (1) add it to analysis_properties.csv, (2) compute its
# stats in the corresponding section above, (3) add one entry to property_stats below.
# The visualization loop adapts automatically.

from analysis.viz_universal import universal_visualizations

# Collect all computed property stats: {pid: (human_label, stats_df)}
# stats_df must have columns: value, person_count, appearance_count
property_stats = {
    "P21":  ("Gender",     c1),
    "P106": ("Occupation", c3),
    "P102": ("Party",      c5),
    "AGE":  ("Age",        c8),
}

print(f"Generating universal visualizations for {len(property_stats)} properties...")
universal_visualizations(property_stats, ALL_DIR, scope="all", top_n=20)


## 14. Guest Frequency Distribution and Pareto Analysis (TASK-B21)

Analyze how many guests appeared exactly 1, 2, 3, ... times, and produce a Pareto chart.

In [ ]:
# Compute guest frequency distribution
guest_appearance_counts = catalogue.groupby('wikidata_id').size().reset_index(name='appearance_count')
freq_dist = guest_appearance_counts['appearance_count'].value_counts().sort_index().reset_index()
freq_dist.columns = ['frequency', 'number_of_guests']

# Pareto: cumulative % of appearances
freq_dist = freq_dist.sort_values('frequency')
freq_dist['appearances'] = freq_dist['frequency'] * freq_dist['number_of_guests']
freq_dist['cumulative_appearances'] = freq_dist['appearances'].cumsum()
total_appearances = freq_dist['appearances'].sum()
freq_dist['pct_cumulative_appearances'] = (freq_dist['cumulative_appearances'] / total_appearances * 100).round(2)

print(f"Total guests: {len(guest_appearance_counts)}")
print(f"Guest frequency distribution:\n{freq_dist.head(10)}")

# TODO: Call viz module for Pareto chart and frequency distribution visualization
# from analysis import plot_guest_frequency_distribution  # (will implement in viz_universal.py)
# fig = plot_guest_frequency_distribution(freq_dist)
# save_fig(fig, data_dir / "50_analysis" / "all" / "guest_frequency_pareto.html")

## 15. Per-Show Analysis (TASK-B22)

For each broadcasting program (show), compute guest count, unique appearances, gender distribution, and top occupations.

In [ ]:
# Per-show analysis -- uses episode_appearances which has show_id from ep_order
guest_ep = episode_appearances[episode_appearances["role"] == "guest"]

per_show_stats = (
    guest_ep
    .groupby(["show_id", "program_name"])
    .agg(
        episode_count=("fernsehserien_de_id", "nunique"),
        guest_appearances=("canonical_entity_id", "count"),
        unique_guests=("canonical_entity_id", "nunique"),
    )
    .reset_index()
)
per_show_stats["avg_guests_per_episode"] = (
    per_show_stats["guest_appearances"] / per_show_stats["episode_count"].clip(lower=1)
).round(2)
per_show_stats = per_show_stats.sort_values("guest_appearances", ascending=False).reset_index(drop=True)

print(f"Per-show statistics ({len(per_show_stats)} shows):")
print(per_show_stats.to_string(index=False))
atomic_write_csv(ALL_DIR / "per_show_statistics.csv", per_show_stats)


## 16. Network & Co-occurrence Analysis (TASK-B10)

Compute guest-to-guest co-occurrence networks and produce interactive visualizations.

In [ ]:
# Co-occurrence analysis -- guests appearing in the same episode
# Uses episode_appearances (one row per person x episode), not catalogue (one row per person)
guest_ep = episode_appearances[episode_appearances["role"] == "guest"]

# Build episode -> list of canonical_entity_ids
episodes_list = (
    guest_ep
    .groupby("fernsehserien_de_id")["canonical_entity_id"]
    .apply(list)
    .values
)

co_occurrence_pairs = []
for episode_guests in episodes_list:
    unique_guests = list(set(episode_guests))
    for i in range(len(unique_guests)):
        for j in range(i + 1, len(unique_guests)):
            guest_a, guest_b = sorted([unique_guests[i], unique_guests[j]])
            co_occurrence_pairs.append({"guest_a": guest_a, "guest_b": guest_b})

if co_occurrence_pairs:
    co_occurrence_df = pd.DataFrame(co_occurrence_pairs)
    co_occurrence_summary = (
        co_occurrence_df
        .groupby(["guest_a", "guest_b"])
        .size()
        .reset_index(name="co_occurrence_count")
        .sort_values("co_occurrence_count", ascending=False)
        .reset_index(drop=True)
    )
    print(f"Total co-occurrence pairs: {len(co_occurrence_summary):,}")
    print(f"Top co-occurrences:")
    print(co_occurrence_summary.head(10).to_string(index=False))
    atomic_write_csv(ALL_DIR / "co_occurrence_pairs.csv", co_occurrence_summary)
else:
    co_occurrence_summary = pd.DataFrame(columns=["guest_a", "guest_b", "co_occurrence_count"])
    print("No co-occurrences found.")


## 17. Export Analysis Results & Summary (TASK-B99)

Write all analysis artifacts to output directory structure: `data/50_analysis/{all,persons,reference,<show_id>/}`

In [ ]:
# Export analysis results
# OUTPUT_DIR is defined in cell 3 as Path("data/50_analysis")

# Directories already created by earlier cells -- just ensure they exist
for dir_path in [ALL_DIR, PERSONS_DIR, REF_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Export already-computed frequency distribution
atomic_write_csv(ALL_DIR / "guest_frequency_distribution.csv", freq_dist)

# Generate analysis summary
n_episodes = episode_appearances["fernsehserien_de_id"].nunique()
n_guests   = len(guests)
n_shows    = len(per_show_stats)

summary_data = {
    "total_episodes_analyzed": n_episodes,
    "total_unique_guests":     n_guests,
    "total_guest_appearances": int(episode_appearances[episode_appearances["role"] == "guest"].shape[0]),
    "average_guests_per_episode": round(
        episode_appearances[episode_appearances["role"] == "guest"].shape[0] / max(n_episodes, 1), 2
    ),
    "broadcasting_programs": n_shows,
    "analysis_timestamp": pd.Timestamp.now().isoformat(),
}

import json as _json
atomic_write_text(_json.dumps(summary_data, indent=2), ALL_DIR / "analysis_summary.json")

print("Analysis export complete!")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Summary: {summary_data}")
